# Entrenar el semáforo predictivo del Monitor V8

Este notebook lee la pestaña **Métricas** de Google Sheets, entrena un modelo fuera de Streamlit y descarga dos archivos:

- `modelo_semaforo.joblib`
- `metricas_modelo.json`

Luego deben subirse a la carpeta `models/` del repositorio. No modifica la planilla.

In [ ]:
!pip -q install gspread pandas scikit-learn joblib

## 1. Iniciar sesión y elegir la planilla

Pegá solamente el ID de la planilla, es decir, lo que aparece entre `/d/` y `/edit` en su URL.

In [ ]:
from google.colab import auth
from google.auth import default
import gspread

SHEET_ID = "PEGAR_AQUI_EL_ID"
PESTANA_METRICAS = "Métricas"

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open_by_key(SHEET_ID)
print("Conectado a:", sh.title)

## 2. Cargar y revisar los datos

El entrenamiento exige al menos 150 notas con título y vistas. Para mejorar el modelo conviene cargar también notas de rendimiento medio, no solamente las mejores y las peores.

In [ ]:
import pandas as pd

rows = sh.worksheet(PESTANA_METRICAS).get_all_records()
df = pd.DataFrame(rows)

for col in ["Titulo", "Seccion", "Fecha", "Hora", "EnPanorama", "Vistas"]:
    if col not in df.columns:
        df[col] = ""

df["Vistas"] = pd.to_numeric(df["Vistas"], errors="coerce")
df = df.dropna(subset=["Vistas"])
df = df[df["Titulo"].astype(str).str.len() >= 8].copy()
print("Notas utilizables:", len(df))
display(df[["Fecha", "Titulo", "Seccion", "Vistas"]].tail(10))

if len(df) < 150:
    raise ValueError("Hay menos de 150 notas. Seguí cargando reportes antes de entrenar.")

## 3. Entrenar y evaluar

El modelo separa el rendimiento en tres niveles relativos: bajo, medio y alto. La prueba utiliza las notas más recientes como conjunto de evaluación.

In [ ]:
import json, math
from datetime import datetime
from pathlib import Path
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Orden temporal cuando la fecha puede interpretarse; si no, conserva el orden de la hoja.
def parse_fecha(value):
    value = str(value).strip()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d/%m"):
        try:
            dt = datetime.strptime(value, fmt)
            if fmt == "%d/%m":
                dt = dt.replace(year=datetime.now().year)
            return dt
        except Exception:
            pass
    return pd.NaT

df["fecha_parseada"] = df["Fecha"].map(parse_fecha)
if df["fecha_parseada"].notna().sum() >= len(df) * 0.6:
    df = df.sort_values("fecha_parseada", na_position="first")

# Etiquetas por terciles de log-vistas.
log_views = (df["Vistas"] + 1).map(math.log10)
c1, c2 = log_views.quantile([1/3, 2/3]).tolist()
def etiqueta(v):
    lv = math.log10(float(v) + 1)
    return "verde" if lv >= c2 else ("amarillo" if lv >= c1 else "rojo")
df["objetivo"] = df["Vistas"].map(etiqueta)

def franja(hora):
    try:
        h = int(str(hora).split(":", 1)[0])
    except Exception:
        return "sin_hora"
    if 5 <= h < 11: return "manana"
    if 11 <= h < 15: return "mediodia"
    if 15 <= h < 20: return "tarde"
    return "noche"

df["titulo"] = df["Titulo"].fillna("").astype(str)
df["seccion"] = df["Seccion"].fillna("sin_seccion").astype(str)
df["franja"] = df["Hora"].map(franja)
df["panorama"] = df["EnPanorama"].fillna("no").astype(str).str.lower().map(
    lambda x: "si" if x in {"sí", "si", "true", "1"} else "no"
)

features = ["titulo", "seccion", "franja", "panorama"]
cut = max(int(len(df) * 0.8), len(df) - 400)
train, test = df.iloc[:cut], df.iloc[cut:]

pre = ColumnTransformer([
    ("titulo", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=6000,
                               strip_accents="unicode", lowercase=True), "titulo"),
    ("categorias", OneHotEncoder(handle_unknown="ignore"), ["seccion", "franja", "panorama"]),
])
model = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(max_iter=1500, class_weight="balanced")),
])
model.fit(train[features], train["objetivo"])
pred = model.predict(test[features])
acc = accuracy_score(test["objetivo"], pred)
f1 = f1_score(test["objetivo"], pred, average="macro")

print(f"Entrenamiento: {len(train)} notas")
print(f"Prueba posterior: {len(test)} notas")
print(f"Accuracy: {acc:.1%}")
print(f"Macro F1: {f1:.1%}")
print(classification_report(test["objetivo"], pred, zero_division=0))

## 4. Publicar los archivos

Ejecutá esta celda y descargá ambos archivos. Después subilos a `models/` en GitHub. Streamlit los cargará en el próximo reinicio.

In [ ]:
from google.colab import files

pack = {
    "format": "v8_pipeline",
    "pipeline": model,
    "thresholds_log10": [float(c1), float(c2)],
    "classes": list(model.classes_),
}
metrics = {
    "fecha_entrenamiento": datetime.now().isoformat(timespec="seconds"),
    "notas_totales": int(len(df)),
    "notas_entrenamiento": int(len(train)),
    "notas_prueba": int(len(test)),
    "accuracy": round(float(acc), 4),
    "macro_f1": round(float(f1), 4),
    "advertencia": "Modelo editorial orientativo; no garantiza audiencia.",
}

joblib.dump(pack, "modelo_semaforo.joblib")
Path("metricas_modelo.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
files.download("modelo_semaforo.joblib")
files.download("metricas_modelo.json")